<!-- NOTEBOOK_OVERVIEW -->
# 1. Feature Engineering and Structural Signal Construction

## 2. Introduction
This notebook develops and inspects the feature families that motivated the later ablation and hybrid experiments. It is primarily an exploratory and sanity-check notebook for lexical, structural, and semantic feature construction rather than the canonical reporting notebook for final dissertation tables.

## 3. Workflow Steps
1. Load the processed Split A dataset and inspect the train/val/test/OOD partitioning.
2. Prototype structural signals such as IBVS v1 and inspect how they distribute across classes and datasets.
3. Build lexical flag features and combine them with sparse lexical text features.
4. Train early baseline models and inspect classification behavior.
5. Generate semantic embeddings used to validate the semantic feature pathway.

## 4. Evaluation and Protocol Notes
1. Metrics shown here are exploratory diagnostics that informed later notebook design.
2. These results are useful for feature intuition and historical comparison, but the dissertation-facing canonical metrics come from notebooks 04, 05, and 06.
3. This notebook is best treated as feature-development evidence rather than the final experimental source of truth.

## 5. Execution Notes
1. Run after preprocessing when you want to inspect intermediate feature behavior or reproduce early development results.
2. Downstream experiments do not depend on this notebook’s runtime state.


In [2]:
# Cell Purpose: Import required libraries, project modules, and shared utilities.
# 1. Imports and configuration

from pathlib import Path
import json
import sys

import numpy as np # data handling
import pandas as pd # data handling

from sklearn.feature_extraction.text import TfidfVectorizer  # lexical features

from sklearn.metrics import classification_report # precision/recall/F1 outputs

from sentence_transformers import SentenceTransformer # semantic embeddings

from xgboost import XGBClassifier # my classifier

from scipy.sparse import hstack, csr_matrix # combinin sparse feature blocks

import re

RANDOM_SEED = 42 # preconfigured random seed for reproducibility
np.random.seed(RANDOM_SEED)


In [3]:
# Cell Purpose: Load input datasets/configuration needed for the next processing stage.
# 2. Load processed dataset v2

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.common.notebook_utils import encode_texts, lexical_flags

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

processed_path = DATA_PROCESSED / "jailbreak_benchmarks_processed_v2.csv"
df = pd.read_csv(processed_path)

print("Rows:", len(df))
print("\nSplit counts:")
print(df["split"].value_counts())
df.head()


Rows: 6424

Split counts:
split
ood_test_injection_standard    3986
ood_test                        768
train                           694
ood_test_injection              678
test                            149
val                             149
Name: count, dtype: int64


,prompt_text,label,dataset_name,attack_family,source_id,text_canonical,distribution_group,split,ood_set,hf_split_name,hf_dataset_id,hf_config_name,source,category,text,trigger_tokens
0,Describe the character of the protagonist in t...,0,alpaca_instructions,benign_instruction,0,describe the character of the protagonist in t...,ood,ood_test,primary,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Describe the effects of exponential growth.,0,alpaca_instructions,benign_instruction,2,describe the effects of exponential growth.,ood,ood_test,primary,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Imagine a future world where robots and humans...,0,alpaca_instructions,benign_instruction,3,imagine a future world where robots and humans...,ood,ood_test,primary,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Create a list of three benefits of cloud compu...,0,alpaca_instructions,benign_instruction,4,create a list of three benefits of cloud compu...,ood,ood_test,primary,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Explain why deep learning is a form of artific...,0,alpaca_instructions,benign_instruction,5,explain why deep learning is a form of artific...,ood,ood_test,primary,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# Cell Purpose: Import required libraries, project modules, and shared utilities.
# 3. Define structural patterns and IBVS v1 (legacy baseline only; IBVS v2 source-of-truth is src/features/ibvs.py)

STRUCTURAL_PATTERNS = {
    "role_redefine": [
        "you are now",
        "from now on you will",
        "pretend to be",
        "act as",
    ],
    "ignore_boundary": [
        "ignore the above",
        "ignore previous instructions",
        "disregard all prior rules",
    ],
    "system_spoof": [
        "<system>",
        "[system]",
        "system:",
    ],
    "meta_override": [
        "as the system prompt",
        "override all policies",
    ],
}

IMPERATIVE_VERBS = [
    "write", "create", "develop", "design", "explain",
    "generate", "produce", "summarise", "summarize",
    "give", "show", "provide", "list", "teach"
]

HARMFUL_ACTION_VERBS = [
    "bypass", "exploit", "manufacture", "build",
    "construct", "smuggle", "hack", "steal",
    "traffic", "fabricate", "weapon", "poison"
]

INSTRUCTION_PATTERNS = [
    r"step[- ]?by[- ]?step",
    r"\bfirst\b",
    r"\bnext\b",
    r"\bthen\b",
    r"\bfinally\b"
]

def ibvs_v1(text: str) -> float:
    """
    Instruction Boundary Violation Score v1
    - Structural overrides (role redefinition, ignoring boundaries, system spoofing, meta override)
    - Imperative starts
    - Harmful action verbs
    - Multi-step instructional structure
    Normalised by token length.
    """
    if not isinstance(text, str):
        return 0.0

    t = text.lower()
    score = 0.0
    
    # Explicit structural overrides
    if any(p in t for p in STRUCTURAL_PATTERNS["role_redefine"]):
        score += 2.0
    if any(p in t for p in STRUCTURAL_PATTERNS["ignore_boundary"]):
        score += 2.0
    if any(p in t for p in STRUCTURAL_PATTERNS["system_spoof"]):
        score += 2.0
    if any(p in t for p in STRUCTURAL_PATTERNS["meta_override"]):
        score += 2.0
    
    # Imperative first word (instruction-like)
    tokens = t.split()
    first_word = tokens[0] if len(tokens) > 0 else ""
    if first_word in IMPERATIVE_VERBS:
        score += 1.0

    # Harmful action verbs anywhere in the text
    if any(v in t for v in HARMFUL_ACTION_VERBS):
        score += 1.5

    # Multi-step instructional frame: "first, next, then, finally" / "step-by-step"
    if any(re.search(p, t) for p in INSTRUCTION_PATTERNS):
        score += 1.0
        
    # Presence of explicitly instructional meta-words
    for cue in ["guide", "instructions", "tutorial", "manual"]:
        if cue in t:
            score += 1.0
            break

    # Length normalisation (avoid simply rewarding long prompts)
    length = max(len(tokens), 1)
    return score / length


In [5]:
# Cell Purpose: Compute IBVS structural features and associated trigger-level diagnostics.
# 4. Attach IBVS v1 to full dataset and sanity-check

df["ibvs_v1"] = df["prompt_text"].apply(ibvs_v1)

print("Global IBVS stats:")
print(df["ibvs_v1"].describe())

print("\nIBVS by label:")
print(df.groupby("label")["ibvs_v1"].describe())

print("\n% non-zero IBVS by label:")
print((df["ibvs_v1"] > 0).groupby(df["label"]).mean())

print("\nMean IBVS by dataset:")
print(df.groupby("dataset_name")["ibvs_v1"].mean().sort_values(ascending=False).head(10))


Global IBVS stats:
count    6424.000000
mean        0.029374
std         0.059905
min         0.000000
25%         0.000000
50%         0.000000
75%         0.021739
max         0.500000
Name: ibvs_v1, dtype: float64

IBVS by label:
        count      mean       std  min  25%       50%       75%       max
label                                                                    
0      2987.0  0.015714  0.037735  0.0  0.0  0.000000  0.008621  0.357143
1      3437.0  0.041244  0.071887  0.0  0.0  0.008197  0.058824  0.500000

% non-zero IBVS by label:
label
0    0.320054
1    0.654641
Name: ibvs_v1, dtype: float64

Mean IBVS by dataset:
dataset_name
advbench                       0.137901
harmbench                      0.094838
jailbreakbench                 0.084954
alpaca_instructions            0.047063
notinject                      0.014253
deepset_prompt_injections      0.013585
qualifire_local                0.008793
r1char9_prompt_injection_v2    0.008007
Name: ibvs_v1, dtype: fl

In [6]:
# Cell Purpose: Import required libraries, project modules, and shared utilities.
# 5. Load fixed splits from processed dataset(v2) and create split-specific views

df_train = df[df["split"] == "train"].copy()
df_val   = df[df["split"] == "val"].copy()
df_test  = df[df["split"] == "test"].copy()
df_ood   = df[df["split"] == "ood_test"].copy()

for name, d in [("train", df_train), ("val", df_val), ("test", df_test), ("ood_test", df_ood)]:
    print(f"{name:8s}", d.shape, d["label"].value_counts().to_dict())


train    (694, 17) {1: 504, 0: 190}
val      (149, 17) {1: 109, 0: 40}
test     (149, 17) {1: 108, 0: 41}
ood_test (768, 17) {0: 384, 1: 384}


In [7]:
# Cell Purpose: Construct lexical feature representations for model training and inference.
# 6. TF–IDF lexical features (unigrams + bigrams)

tfidf = TfidfVectorizer(
    ngram_range=(1, 2),      # surface lexical patterns: words + short phrases
    min_df=2,                # drop ultra-rare terms
    max_features=20000,      # keep feature space manageable
)

tfidf.fit(df_train["prompt_text"])  # IMPORTANT: fit on train only

X_lex_train = tfidf.transform(df_train["prompt_text"])
X_lex_val   = tfidf.transform(df_val["prompt_text"])
X_lex_test  = tfidf.transform(df_test["prompt_text"])
X_lex_ood   = tfidf.transform(df_ood["prompt_text"])

y_train = df_train["label"].values
y_val   = df_val["label"].values
y_test  = df_test["label"].values
y_ood   = df_ood["label"].values

X_lex_train.shape, X_lex_val.shape, X_lex_test.shape, X_lex_ood.shape


((694, 2157), (149, 2157), (149, 2157), (768, 2157))

In [8]:
# Cell Purpose: Construct lexical feature representations for model training and inference.
# 7. Lexical flag features (override cues, length, etc.)

OVERRIDE_PATTERNS = [
    r"ignore (all )?(previous|prior) instructions",
    r"you are now",
    r"disregard (the )?(previous|above) rules",
    r"as an unfiltered model",
    r"system prompt",
    r"from now on, you must",
]

lex_flags_train = pd.DataFrame([
    lexical_flags(t, override_patterns=OVERRIDE_PATTERNS) for t in df_train["prompt_text"]
])
lex_flags_val = pd.DataFrame([
    lexical_flags(t, override_patterns=OVERRIDE_PATTERNS) for t in df_val["prompt_text"]
])
lex_flags_test = pd.DataFrame([
    lexical_flags(t, override_patterns=OVERRIDE_PATTERNS) for t in df_test["prompt_text"]
])
lex_flags_ood = pd.DataFrame([
    lexical_flags(t, override_patterns=OVERRIDE_PATTERNS) for t in df_ood["prompt_text"]
])

lex_flags_train.head()


,has_ignore_prev,has_you_are_now,has_disregard,has_unfiltered,has_system_prompt,has_from_now_on,len_chars,len_tokens_approx
0,False,False,False,False,False,False,81,13
1,False,False,False,False,False,False,92,13
2,False,False,False,False,False,False,45,8
3,False,False,False,False,False,False,74,13
4,False,False,False,False,False,False,97,18


In [9]:
# Cell Purpose: Compute IBVS structural features and associated trigger-level diagnostics.
# 8. Integrate IBVS into structural feature matrices

# Add IBVS numeric feature into the lexical flag tables
lex_flags_train["ibvs_v1"] = df_train["ibvs_v1"].values
lex_flags_val["ibvs_v1"]   = df_val["ibvs_v1"].values
lex_flags_test["ibvs_v1"]  = df_test["ibvs_v1"].values
lex_flags_ood["ibvs_v1"]   = df_ood["ibvs_v1"].values

# Convert the DataFrames into sparse matrices (for hstack with TF–IDF)
# Compressed Sparse Row (CSR) matrix in machine learning is a specialised data 
# structure used to efficiently store and manipulate sparse matrices—large, 
# high-dimensional matrices where most elements are zero.

X_struct_train = csr_matrix(lex_flags_train.values.astype(float))
X_struct_val   = csr_matrix(lex_flags_val.values.astype(float))
X_struct_test  = csr_matrix(lex_flags_test.values.astype(float))
X_struct_ood   = csr_matrix(lex_flags_ood.values.astype(float))

X_struct_train.shape, X_struct_val.shape


((694, 9), (149, 9))

In [10]:
# Cell Purpose: Compute IBVS structural features and associated trigger-level diagnostics.
# 9. Fuse lexical TF–IDF and structural features (incl. IBVS)
# Essential for preprocessing data by merging different feature sets,
# such as combining numerical and encoded categorical data into a single, 
# wider matrix for model training. 

X_train_fused = hstack([X_lex_train, X_struct_train]).tocsr()
X_val_fused   = hstack([X_lex_val,   X_struct_val]).tocsr()
X_test_fused  = hstack([X_lex_test,  X_struct_test]).tocsr()
X_ood_fused   = hstack([X_lex_ood,   X_struct_ood]).tocsr()

X_train_fused.shape, X_val_fused.shape, X_test_fused.shape, X_ood_fused.shape


((694, 2166), (149, 2166), (149, 2166), (768, 2166))

In [11]:
# Cell Purpose: Train model(s) using the prepared feature sets and split configuration.
# 10. Baseline XGBoost classifier on fused features
# XGBoost handles sparse input well, can learn non-linear interactions, and has
# a strong baseline in tabular and sparse hybrid settings
xgb = XGBClassifier(
    objective="binary:logistic",
    n_estimators=400,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.9,
    colsample_bytree=0.9,
    eval_metric="logloss",
    n_jobs=-1,
    random_state=RANDOM_SEED,
)

xgb.fit(
    X_train_fused,
    y_train,
    eval_set=[(X_val_fused, y_val)],
    verbose=False,
)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.9, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=400, n_jobs=-1,
              num_parallel_tree=None, ...)

In [12]:
# Cell Purpose: Configure and validate split-specific data partitions and runtime settings.
# 11. Evaluation helper and reports (VAL / TEST / OOD)

def evaluate_split(name, X, y_true):
    y_pred = xgb.predict(X)
    y_proba = xgb.predict_proba(X)[:, 1]
    print(f"\n=== {name} ===")
    print(classification_report(y_true, y_pred, digits=3))
    return y_pred, y_proba

y_val_pred,  y_val_proba  = evaluate_split("VAL",  X_val_fused,  y_val)
y_test_pred, y_test_proba = evaluate_split("TEST", X_test_fused, y_test)
y_ood_pred,  y_ood_proba  = evaluate_split("OOD",  X_ood_fused,  y_ood)



=== VAL ===
              precision    recall  f1-score   support

           0      0.853     0.725     0.784        40
           1      0.904     0.954     0.929       109

    accuracy                          0.893       149
   macro avg      0.879     0.840     0.856       149
weighted avg      0.891     0.893     0.890       149


=== TEST ===
              precision    recall  f1-score   support

           0      0.867     0.634     0.732        41
           1      0.874     0.963     0.916       108

    accuracy                          0.872       149
   macro avg      0.870     0.799     0.824       149
weighted avg      0.872     0.872     0.866       149


=== OOD ===
              precision    recall  f1-score   support

           0      0.673     0.602     0.635       384
           1      0.640     0.708     0.672       384

    accuracy                          0.655       768
   macro avg      0.657     0.655     0.654       768
weighted avg      0.657     0.655 

In [13]:
# Cell Purpose: Build or load semantic feature representations for baseline/hybrid models.
# 12. Semantic embeddings with BAAI/bge-small-en-v1.5
# NOTE: OOD embeddings are computed for reporting/analysis only. 
# No thresholds, calibration, or model parameters are selected using OOD.

sem_model = SentenceTransformer("BAAI/bge-small-en-v1.5")

X_sem_train = encode_texts(sem_model, df_train["prompt_text"].tolist(), show_progress_bar=True)
X_sem_val   = encode_texts(sem_model, df_val["prompt_text"].tolist(), show_progress_bar=True)
X_sem_test  = encode_texts(sem_model, df_test["prompt_text"].tolist(), show_progress_bar=True)
X_sem_ood   = encode_texts(sem_model, df_ood["prompt_text"].tolist(), show_progress_bar=True)

X_sem_train.shape, X_sem_val.shape


Batches:   0%|          | 0/22 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/24 [00:00<?, ?it/s]

((694, 384), (149, 384))

<!-- NOTEBOOK_OUTPUT_SUMMARY -->
## 6. Output Summary
1. Produces exploratory feature diagnostics for lexical, IBVS, and semantic representations.
2. Documents early baseline model behavior that informed the final ablation and hybrid design.
3. Serves as historical feature-development evidence rather than the primary source of dissertation result tables.
